# Module 32 — Exercise 1: Worker Concurrency Models & Throughput Measurement

In this exercise, you will measure and compare the latency and throughput profile of synchronous thread pools vs asynchronous event loops for I/O-bound and CPU-bound workloads.

| Detail | Value |
|---|---|
| **Time** | 35 minutes |
| **Prerequisites** | Module 32 README, Modules 21, 22 |



## 1. Concurrency Benchmarking Harness


In [ ]:
import asyncio
import time
from concurrent.futures import ThreadPoolExecutor

async def simulate_io_async(delay: float) -> None:
    await asyncio.sleep(delay)

def simulate_io_sync(delay: float) -> None:
    time.sleep(delay)



# Your turn


### Task 1: Measure Async vs ThreadPool Completion Time

Implement `benchmark_concurrency(num_tasks, task_duration)` which executes `num_tasks` simulated I/O tasks using:
1. `asyncio.gather`
2. `ThreadPoolExecutor(max_workers=10)`
Return a tuple `(async_time, thread_time)`.


In [ ]:
# ANSWER 1
async def _run_async(n: int, d: float) -> float:
    t0 = time.perf_counter()
    await asyncio.gather(*(simulate_io_async(d) for _ in range(n)))
    return time.perf_counter() - t0

def _run_threads(n: int, d: float, workers: int = 10) -> float:
    t0 = time.perf_counter()
    with ThreadPoolExecutor(max_workers=workers) as pool:
        list(pool.map(simulate_io_sync, [d] * n))
    return time.perf_counter() - t0

async def benchmark_concurrency(num_tasks: int, task_duration: float):
    t_async = await _run_async(num_tasks, task_duration)
    t_threads = _run_threads(num_tasks, task_duration, workers=10)
    return (t_async, t_threads)



## Self-Check Harness


In [ ]:
def check(passed: bool, msg: str) -> bool:
    status = "PASS" if passed else "FAIL"
    print(f"{status}  {msg}")
    return passed

# Test with 20 tasks of 0.05s
async def main_check():
    t_async, t_threads = await benchmark_concurrency(20, 0.02)
    print(f"Async duration: {t_async:.3f}s | ThreadPool(10) duration: {t_threads:.3f}s")
    results = [
        check(t_async < 0.15, "Task 1: Async executed 20 I/O tasks concurrently in < 0.15s"),
        check(t_threads > t_async, "Task 1: Thread pool bounded by worker count takes longer than unbounded coroutines"),
    ]
    print(f"Summary: {sum(results)}/{len(results)} checks passed.")

await main_check()

